![AgentwithMemory](./AgentwithMemory.png)

### Agent

<h6>
Our chat model(LLM) will decide to make a tool call or not based upon the user input<br>
We use a conditial edge to route to a node that will call our tool or simply end
</h6>

<h6>
Pass <b>ToolMessage</b> back to the model.<br>
We can let it either (1) call another tool or (2) respond directly. 
This is the intuition behind ReAct, a general agent architecture.
<ul>   
<li>act - let the model call specific tools</li>
<li>observe - pass the tool output back to the model</li>
<li>reason - let the model reason about the tool output to decide what to do next (e.g., call another tool or just respond directly)</li>
</ul>   
</h6>

### Router

<h6>
Graph uses <b>messages</b> as state and chat model with bound tools.<br>
The Chat model routes between a direct response or a tool call based upon the user input.
</h6>

In [ ]:
import os


def load_env():
    """Load environment variables from the local .env file."""
    env_path = "C:\\Users\\felip\\Documents\\introduction_langgraph\\.env"

    with open(env_path, "r") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_env()

### Tools

<h6>
Tools are needed whenever you want a model to control parts of your code or call out to external API.<br>
Pass any Python function into <b>ChatModel.bind_tools()</b>
</h6>

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

def divide(a: int, b: int) -> int:
    """Divide two integers and return the result."""
    return a / b

tools = [add, multiply, divide]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY")) # gemini-2.5-flash
llm_tools = llm.bind_tools(tools)

### Chat Models
<h6>
Chat models can use a sequence of message as input and support message roles, as discussed above.<br>
</h6>

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage


# System Message
system_message = SystemMessage(content="You are a helpful assistant that can answer questions about ocean mammals.")

# Node
def assistant(state: MessagesState) -> AIMessage:
    """Simulate an assistant node that processes messages."""
    # For demonstration, we will just return a simple response.
    return {"messages": [llm_tools.invoke([system_message] + state["messages"])]}

### ToolNode
<h6>
The built-in <b>tools_condition</b> is used as a conditional edge.<br>
</h6>

In [ ]:
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import MessagesState, StateGraph, START, END


# Graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

display(Image(react_graph.get_graph().draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 5.")]
messages = react_graph.invoke({"messages": messages})

In [ ]:
for message in messages["messages"]:
    print(f"{message.type}: {message.content}")

### Memory
<h6>

</h6>

In [ ]:
messages = [HumanMessage(content="Add 3 and 4.")]
messages = react_graph.invoke({"messages": messages})

for message in messages["messages"]:
    message.pretty_print()

In [ ]:
messages = [HumanMessage(content="Multiply that by 2.")]
messages = react_graph.invoke({"messages": messages})

for message in messages["messages"]:
    message.pretty_print()

<h6>
The <b>state</b> is transient to a single graph execution.<br>
This limits our ability to have multi-turn conversations with interruptions.<br>
LangGraph can use a checkpointer to automatically save the graph state after each step.<br>
This built-in persistence layer gives us memory, allowing LangGraph to pick up from the last state update.<br>
One of the easiest checkpointers to use is the <b>MemorySaver</b>, an in-memory key-value store for Graph state.<br>
Compile the graph with a checkpointer, and our graph has memory.<br>
</h6>

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
react_graph_memory = builder.compile(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

messages = [HumanMessage(content="Add 3 and 4.")]

messages = react_graph_memory.invoke({"messages": messages, "config": config})

for message in messages["messages"]:
    message.pretty_print()

In [ ]:
messages = [HumanMessage(content="Multiply that by 2.")]

messages = react_graph_memory.invoke({"messages": messages, "config": config})

for message in messages["messages"]:
    message.pretty_print()